# Eksploracja danych · Laboratorium 5
## Wykrywanie anomalii (spotkanie 2): LOF, porównanie metod, koszt decyzji, walidacja, dryf

Od tego spotkania funkcje są w module `anomalie.py` (projekt zaliczeniowy). Notatnik leży w katalogu projektu, obok `anomalie.py` i `dane/`.
Najpierw uzupełnij funkcje z TODO i uruchom `pytest -q` w terminalu.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anomalie as A

S = A.dodaj_cechy(A.wczytaj("dane/dostawy_styczen.csv"))
L = A.dodaj_cechy(A.wczytaj("dane/dostawy_luty.csv"))
M = A.dodaj_cechy(A.wczytaj("dane/dostawy_marzec.csv"))
y = S.etykieta.to_numpy()
BUDZET = 60
pd.set_option("display.precision", 4)

def r(m):
    """Metryki do wydruku: liczby zaokrąglone do 4 miejsc."""
    return {k: (int(v) if k in ("alarmy", "TP", "FP", "FN", "TN", "koszt") else round(float(v), 4)) for k, v in m.items()}

## Z1. LOF i porównanie pięciu metod przy tym samym budżecie alarmów

In [ ]:
mk = A.model_kosztu(S)
norma = A.norma_reguly(S, mk)
X, skaler = A.macierz_cech(S)
W = {
    "IQR koszt": S.koszt_pln.to_numpy(),
    "Model normy": A.wynik_reguly(S, mk, norma).to_numpy(),
    "K-means": A.wynik_kmeans(X, 5)[0],
    "Isolation Forest": A.wynik_isolation_forest(X)[0],
    "LOF": A.wynik_lof(X, 20),
}
tab = pd.DataFrame({k: r(A.metryki(y, A.alarm_top(v, BUDZET))) for k, v in W.items()}).T
tab

In [ ]:
typy = pd.DataFrame({k: A.recall_wg_typu(S, A.alarm_top(v, BUDZET)) for k, v in W.items()}).T
typy

**Pytania:** (1) Która metoda wygrywa i dlaczego (pomyśl o tym, co wie o procesie)? (2) Dlaczego LOF łapie „koszt +1600 zł” lepiej niż Isolation Forest?

*Odpowiedź:* …

## Z2. Ile alarmów sprawdzać? Krzywa kosztu

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for k, v in W.items():
    c = A.krzywa_kosztu(v, y, n_max=400)
    ax.plot(c.alarmy, c.koszt / 1000, label=k)
    i = c.koszt.idxmin()
    print(f"{k:18s} najniższy koszt {c.koszt[i]:8.0f} zł przy {int(c.alarmy[i])} alarmach (TP = {int(c.TP[i])})")
ax.axvline(BUDZET, ls=":", color="k"); ax.set_xlabel("liczba alarmów"); ax.set_ylabel("koszt [tys. zł]"); ax.legend(); plt.show()

**Pytanie:** krzywą policzyliśmy z etykietami. Dlaczego „optymalna liczba alarmów” ze stycznia to optymistyczne oszacowanie i jak to sprawdzić?

*Odpowiedź:* …

## Z3. Walidacja na lutym: modele, skaler i progi ze stycznia

In [ ]:
XL, _ = A.macierz_cech(L, skaler=skaler)
km_model = A.wynik_kmeans(X, 5)[1]; if_model = A.wynik_isolation_forest(X)[1]
WL = {
    "IQR koszt": L.koszt_pln.to_numpy(),
    "Model normy": A.wynik_reguly(L, mk, norma).to_numpy(),
    "K-means": A.wynik_kmeans(XL, model=km_model)[0],
    "Isolation Forest": A.wynik_isolation_forest(XL, model=if_model)[0],
    "LOF": A.wynik_lof(XL, 20, X_uczenie=X),
}
wal = {}
for k in W:
    prog = np.sort(W[k])[::-1][BUDZET - 1]          # próg = wynik 60. dostawy w styczniu
    wal[k] = r(A.metryki(L.etykieta, A.alarm_prog(WL[k], prog - 1e-9)))
pd.DataFrame(wal).T

In [ ]:
print("Model normy, stały próg z > 3,5:", r(A.metryki(L.etykieta, WL["Model normy"] > 3.5)))

## Z4. Marzec: nowy cennik od 16.03 (dryf procesu)

In [ ]:
M["alarm"] = (A.wynik_reguly(M, mk, norma) > 3.5).to_numpy()
po = (M.data >= "2026-03-16").to_numpy()
dzien = M.groupby("data").alarm.sum()
print("alarmów dziennie przed 16.03:", round(dzien[dzien.index < "2026-03-16"].mean(), 2), " po:", round(dzien[dzien.index >= "2026-03-16"].mean(), 2))
print("przed:", r(A.metryki(M.etykieta[~po], M.alarm[~po])))
print("po:   ", r(A.metryki(M.etykieta[po], M.alarm[po])))
dzien.plot(kind="bar", figsize=(11, 3.5)); plt.ylabel("alarmy dziennie"); plt.show()

In [ ]:
M["km_po_podwyzce"] = M.odleglosc_km * po          # stawka za km zmienia się od 16.03
mkM = A.model_kosztu(M, A.CECHY_MODELU + ["km_po_podwyzce"])
print("stawka za km przed:", round(mkM.coef_[0], 3), " dodatek po 16.03:", round(mkM.coef_[-1], 3))
M["alarm2"] = (A.wynik_reguly(M, mkM, A.norma_reguly(M, mkM)) > 3.5).to_numpy()
d2 = M.groupby("data").alarm2.sum()
print("po aktualizacji modelu: alarmów dziennie po 16.03:", round(d2[d2.index >= "2026-03-16"].mean(), 2))
print("po 16.03:", r(A.metryki(M.etykieta[po], M.alarm2[po])))

**Pytania:** (1) Czy model „wykrył zmianę procesu”? Czym różni się alarm dla pojedynczej dostawy od sygnału, że zmieniła się norma? (2) Jak często aktualizować model i kto powinien o tym zdecydować?

*Odpowiedź:* …

## Z5. Projekt: testy, dashboard, GitHub, Render

W terminalu, w katalogu projektu:
```
pytest -q
streamlit run app.py
```
Potem kroki z `README.md`: repozytorium na GitHubie, wdrożenie na Render. Do README dopisz tabelę z Z1 (budżet 60), trzy wnioski ekonomiczne i jedno ograniczenie analizy.